# レコメンドアルゴリズム：個人の趣向に沿った商品をお薦めする

・おもちゃ会社勤務  
・主にECサイト経由でおもちゃを販売  
・新商品の売り上げが伸びないため、新商品のPRに力を入れる  
・要望：機械学習で新商品の売り上げアップのための施策を企画したい  

## フェーズA：ビジネス課題分析

### 解決したい課題は何か

【背景】  
・ECサイトで商品を買うには、会員登録の必要があり、会員に販促メールを送ることが出来る。  
・しかし、会社には子供向けから大人向け、男性向け女性向けの商品というような様々なジャンルを販売している関係で、毎月大量の新商品が販売されるため、新しい商品が出ても知らない顧客が多い。  
・また、同様の理由から、新しい商品が出るたびに会員全員に販促メールを送ることは現実的ではない。  
・会社の商品には愛着を持ってくれいている顧客が多く、販促メールが適切に送られれば購入率は低くないと予想される。  

→　`新商品を買ってくれそうなユーザーの予測`が出来れば、新商品の販促メールを特定のユーザーに対して送信でき、売り上げアップにつながりそう！

### 予測したい値は何か？どんなアクションに使えるか？

『新商品を買ってくれそうなユーザーの予測』  

→ `レコメンドアルゴリズム`：ユーザーの過去の購入履歴などから商品をレコメンドするアルゴリズム  
→ これにより、「各ユーザーに対しての新商品のおすすめの度合いを予測」できそう！

～分類アルゴリズムではなく、レコメンドアルゴリズムを用いるメリット～    
分類アルゴリズムで「購入するか、しないか」を分類するモデルを構築することも可能！  
だが、ここでレコメンドアルゴリズムを用いるメリットは...  

1. `複雑な相互作用`  
　レコメンドアルゴリズムは、ユーザーとアイテムの相互作用や関係性を考慮する必要がある。
2. `スパース性`  
　今回のようなケースでは、ユーザーとアイテムの相互作用の行列は大部分が欠損している場合が多い(=スパース性)。レコメンドアルゴリズムではこのスパース性を効果的に取り扱える。  
3. `順位付け`  
　レコメンデーションタスクでは、商品をユーザーに対して順位付けすることが求められることが多いが、分類アルゴリズムでは難しい。
5. `コールドスタート問題`
　新しいユーザーや商品が追加された際、そのユーザーや商品に対する過去のデータが不足していることが多いが、このような問題もレコメンドアルゴリズムは効果的に取り扱える。  


## フェーズB：データ分析、機械学習  

### データ観察

In [1]:
import pandas as pd
df = pd.read_csv("animal_rating.csv", index_col=0)
df.head()

,犬,猫,ペンギン,イルカ,パンダ
user_1,NaN,5.0,NaN,5.0,NaN
user_2,NaN,NaN,5.0,NaN,2.0
user_3,NaN,5.0,5.0,NaN,5.0
user_4,4.0,NaN,NaN,4.0,5.0
user_5,5.0,4.0,NaN,NaN,5.0


行データ：ユーザー  
列データ：商品名  
値：レビュースコア  

これまでは、テーブルデータ内の特定の特徴量を「予測したい値」として予測を行ってきたが、  
今回は特定の列を予測するのではなく、まだ評価が行われていない(NaN)商品に対してのレビュースコアを予測する！  
（レコメンドにも様々なバリエーションがあり、これは最もよく知られる一般的な形式：レビュースコアマトリックス）   

### 特徴量エンジニアリング

レコメンドアルゴリズム専用のライブラリである`surprise`を用いる。  
（scikit-learnではレビュースコアマトリックスに対してのレコメンドは行えない）  

レコメンドアルゴリズムで用いるデータは`疎なデータ`(欠損値が多いデータ)であることが多い。  
疎なデータではデータサイズが無駄に大きくなる傾向がある。  
（例：100個の商品に対してレビューを行っているのが1つだけだと学習に使えるのは1つだけなのに計算量だけ多くなる）  

`surprise`では疎なデータを取り扱うためのデータ形式が求められるため、pandasのDataFrameの形式ではinputできない。  
→ 列もちのデータを`行もち`にする！

In [2]:
df_stacked = df.stack().dropna().to_frame().reset_index()
df_stacked.columns = ["ユーザーID", "商品ID", "rating"]
df_stacked.head()

,ユーザーID,商品ID,rating
0,user_1,猫,5.0
1,user_1,イルカ,5.0
2,user_2,ペンギン,5.0
3,user_2,パンダ,2.0
4,user_3,猫,5.0


疎なデータでは、行もちのデータに変換することで、データ容量の観点や計算効率の観点でメリットを享受できる！  

この後、以下の通り、surpriseのDataset形式に変換する必要がある。  

In [3]:
# pip install scikit-surprise

In [4]:
from surprise import Dataset, Reader

# rating_scale：ratingの範囲設定。今回レビュースコアが1~5なので(1,5)としている。
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_stacked, reader)

### アルゴリズムと評価指標の選定

【レコメンドタスクで用いられる主要なアルゴリズム】  
・`SVD (Singular Value Decomposition;特異値分解)`  
　大きな行列を特性を保った小さな行列へ分解する手法の一つ。大量のユーザーやアイテム情報を短い特徴ベクトル(潜在因子)に圧縮し、ユーザーとアイテム間の関係性を見つけ出す。  
　具体的には、ユーザーとアイテム間の評価行列(例：アイテムに着けた評価スコアなど)を分解し、未評価のアイテムに対するユーザーの評価を予測する。SVDは、大きな行列を扱いやすい形に変換し、欠損値の予測や次元削減などにも用いられる。  
　計算効率が良く、潜在因子という考え方でアイテム間の関係性を上手く捉えられるため、昔から多くの推薦システムで用いられる。  

・`KNN (k-Nearest Neighbors)`  
　アイテムやユーザー間の類似性を計算し、最も類似性の高いk個のアイテムやユーザーから推薦を行うアルゴリズム。 

・`NMF (Non-Negative Matrix Factorization)`  
　正の値しか持たない行列(例：ユーザー/アイテムの評価行列)を2つの正の行列に分解し、欠損値を補完するアルゴリズム。  
　SVDと同様、行列分解を行うが、全ての値が正であるという制約がある。  
 
・`MF (Matrix Factorization)`  
　SVDと似ているが、欠損値が存在する行列に対しても適用可能。  
　一般的には潜在的な特徴(ユーザーの隠れた好みやアイテムの隠れた特性など)を発見するために使用する。  
 
・`Deep Learningを用いた手法`  
　Autoencoderを用いた手法や、ニューラルネットワークを用いた協調フィルタリングなどがある。

### モデルの学習

#### データ分割

In [5]:
from surprise.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.25)

#### 学習

今回は`SVD`を用いる。  
scikit-learnと同じくfitメソッドを呼び出すことで学習できる。  

In [6]:
from surprise import SVD

model = SVD()
model.fit(train)

#### モデルの評価

評価指標は`RMSE`を用いる。  
レコメンドシステムにおけるRMSEは、ユーザーがアイテムに対して付ける評価の予測値と実際の予測値との差を二乗したものの平均を計算し、最後に平方根を取った値。  
RMSEが低いほど、モデルが予測した評価値が実際の評価値に近いという意味。

In [7]:
from surprise import accuracy

test_pred = model.test(test)
accuracy.rmse(test_pred)

RMSE: 1.2004


np.float64(1.2003611975064887)

【その他の評価指標】  
1. `MAE(Mean Absolute Error)`  
　予測と実際の評価との差の絶対値の平均。個々の誤差を等しく扱うため、外れ値の影響を受けに行くいという特性がある。

2. `Precision@k and Recall@k`  
　k個の推薦されたアイテムの中で、ユーザーが本当に好きだったアイテムの割合(Precision)と、ユーザーが好きなアイテムのうち、どれだけを推薦できたか(Recall)を計算する。
 
3. `F-Score`  
　PrecisionとRecallの調和平均を計算する。PrecisionとRecallのバランスを取るための指標で、どちらも重視する場合に使われる。

### モデルを使った予測

In [8]:
# user_70というユーザーへのレコメンドを取得する  
user_id = "user_70"
print(f"{user_id}に対するおすすめ度")

for item_id in ["犬", "猫", "ペンギン", "イルカ", "パンダ"]:
    pred = model.predict(user_id, item_id)
    print(f"{item_id}\t{pred.est:.2},")

user_70に対するおすすめ度
犬	4.1,
猫	3.8,
ペンギン	4.6,
イルカ	4.2,
パンダ	3.6,


この中で「user_70がまだ購入していない」「最もおすすめ度が高い」ものを選んで、販促メールを送信できそう。